In [1]:
# %load_ext autoreload
# %autoreload 2

# from utgen.raggraph.utils import get_node_context
# from utgen.test_generation_crew.crew import TestGenerationCrew

In [41]:
import json

# Open the file in read mode ('r')
with open('../data/tests_data.json', 'r', encoding='utf-8') as f:
    # Load the file content into a Python dictionary
    data = json.load(f)

In [42]:
for k, v in data['tests'].items():
    print(k, v)
    print()

test_iter_python_files_basic {'name': 'test_iter_python_files_basic', 'imports': 'import pytest\nfrom unittest.mock import Mock, patch\nfrom pathlib import Path', 'code': "@patch('os.walk')\ndef test_iter_python_files_basic(mock_walk):\n    mock_walk.return_value = [\n        ('/test/path', [], ['file1.py', 'file2.py', '__init__.py'])\n    ]\n    \n    result = list(iter_python_files('/test/path'))\n    \n    assert result == ['/test/path/file1.py', '/test/path/file2.py']\n    mock_walk.assert_called_once_with('/test/path')"}

test_iter_python_files_skip_init_false {'name': 'test_iter_python_files_skip_init_false', 'imports': 'import pytest\nfrom unittest.mock import Mock, patch\nfrom pathlib import Path', 'code': "@patch('os.walk')\ndef test_iter_python_files_skip_init_false(mock_walk):\n    mock_walk.return_value = [\n        ('/test/path', [], ['file1.py', '__init__.py'])\n    ]\n    \n    result = list(iter_python_files('/test/path', skip_init=False))\n    \n    assert result == ['

In [45]:
import subprocess
import os

def validar_test_individual(import_code, test_code):
    """
    Prova un test en un fitxer temporal. 
    Retorna True si el test passa, False si no.
    """
    temp_filename = "temp_validation.py"
    full_code = f"{import_code}\n\n{test_code}"

    with open(temp_filename, "w") as f:
        f.write(full_code)
    
    # Executem pytest. -q per silenci, --tb=no per no embrutar la consola
    result = subprocess.run(["pytest", temp_filename, "-q", "--tb=no"], capture_output=True)
    
    if os.path.exists(temp_filename):
        os.remove(temp_filename)
        
    return result.returncode == 0

def guardar_i_netejar_tests(tests_valids, desti="../tests/test_generats_llm.py"):
    """
    tests_valids: Llista de tuples [(import, codi), ...]
    """
    if not tests_valids:
        print("No hi ha tests vàlids per guardar.")
        return

    os.makedirs(os.path.dirname(desti), exist_ok=True)

    # 1. SEPAREM EN DOS BLOCS
    bloc_imports = []
    bloc_tests = []
    
    for imp, code in tests_valids:
        bloc_imports.append(imp.strip())
        bloc_tests.append(code.strip())
    
    # 2. CONCATENEM: primer TOTS els imports, després els tests
    contingut_final = "\n".join(bloc_imports) + "\n\n" + "\n\n".join(bloc_tests)

    with open(desti, "w") as f:
        f.write(contingut_final)

    # 2. Passem Ruff per endreçar-ho tot
    print(f"Netejant {desti} amb Ruff...")
    
    # Ordenar imports i eliminar duplicats (isort)
    subprocess.run(["ruff", "check", "--select", "I", "--fix", desti], capture_output=True)
    # # Eliminar variables/imports no usats i altres errors comuns
    subprocess.run(["ruff", "check", "--fix", desti], capture_output=True)
    # # Formatar codi (estil Black)
    subprocess.run(["ruff", "format", desti], capture_output=True)
    
    print(f"✅ Procés finalitzat. Fitxer guardat a: {desti}")

In [46]:
node_id = "../src/utgen/raggraph/walker.py::function::iter_python_files"

split_1 = node_id.split('/')
split_2 = split_1[-1].split('::')
split_3 = split_2[0].split('.')

test_func_import = f"\nfrom {split_1[2]}.{split_1[3]}.{split_3[0]} import {split_2[-1]}"

In [47]:
test_func_import

'\nfrom utgen.raggraph.walker import iter_python_files'

In [48]:
tests_que_han_passat = []

for k, v in data['tests'].items():
    name, imp, code = k, v['imports'], v['code']
    imp = imp + test_func_import
    if validar_test_individual(imp, code):
        print(f"✅ Test `{name}` acceptat")
        tests_que_han_passat.append((imp, code))
    else:
        print(f"❌ Test `{name}` rebutjat")

# Guardem i deixem el fitxer perfecte
guardar_i_netejar_tests(tests_que_han_passat)

✅ Test `test_iter_python_files_basic` acceptat
✅ Test `test_iter_python_files_skip_init_false` acceptat
✅ Test `test_iter_python_files_no_python_files` acceptat
✅ Test `test_iter_python_files_nested_directories` acceptat
Netejant ../tests/test_generats_llm.py amb Ruff...
✅ Procés finalitzat. Fitxer guardat a: ../tests/test_generats_llm.py
